# LLM 사용 과제2 + 과제3 통합 파이프라인

In [ ]:
from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union
import json, os, re
import pandas as pd


EXCEL_PATH = Path("consistency_check.xlsx")
MD_PATH = Path("task3_nl_to_modules.md")
OUT_DIR = Path("llm_pipeline_output")

USE_REAL_LLM = True
MODEL_NAME = "gpt-5.5"
MAX_RETRY = 5
os.environ["OPENAI_API_KEY"] = ""

if not EXCEL_PATH.exists() and Path("/mnt/data/consistency_check.xlsx").exists():
    EXCEL_PATH = Path("/mnt/data/consistency_check.xlsx")

if not MD_PATH.exists() and Path("/mnt/data/task3_nl_to_modules.md").exists():
    MD_PATH = Path("/mnt/data/task3_nl_to_modules.md")

OUT_DIR.mkdir(exist_ok=True)

## 1. 과제2 Excel → equipment_data 생성

In [2]:
def clean_value(value: Any) -> Any:
    if pd.isna(value):
        return None
    return value


def to_float_or_none(value: Any) -> Optional[float]:
    value = clean_value(value)
    if value is None or value == "":
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def yes_no_to_bool(value: Any) -> Optional[bool]:
    value = clean_value(value)
    if value is None:
        return None
    text = str(value).strip().upper()
    if text in {"Y", "YES", "TRUE", "1"}:
        return True
    if text in {"N", "NO", "FALSE", "0"}:
        return False
    return None


def normalize_class_name(raw_type: Any) -> str:
    text = str(clean_value(raw_type) or "").strip().lower()
    if "tank" in text:
        return "Tank"
    if "pump" in text:
        return "Pump"
    if "heat" in text and "exchanger" in text:
        return "HeatExchanger"
    if "vessel" in text:
        return "Vessel"
    if "compressor" in text:
        return "Compressor"
    return "Equipment"


def read_excel_safely(excel_path: Path) -> Dict[str, pd.DataFrame]:
    return pd.read_excel(excel_path, sheet_name=None)


def build_equipment_data(excel_path: Path):
    sheets = read_excel_safely(excel_path)
    eq_raw = sheets["Equipment_List"].copy()
    line_raw = sheets["Line_List"].copy()

    equipment_data = []

    for _, row in eq_raw.iterrows():
        tag = clean_value(row.get("Equipment Tag"))
        if tag is None:
            continue

        equipment_data.append({
            "Tag": str(tag).strip(),
            "OriginalType": clean_value(row.get("Type")),
            "ClassName": normalize_class_name(row.get("Type")),
            "OperatingPressure": to_float_or_none(row.get("Operating Pressure (barg)")),
            "DesignPressure": to_float_or_none(row.get("Design Pressure (barg)")),
            "OperatingTemperature": to_float_or_none(row.get("Operating Temp (degC)")),
            "DesignTemperature": to_float_or_none(row.get("Design Temp (degC)")),
            "Material": clean_value(row.get("Material")),
            "HasSafetyValve": yes_no_to_bool(row.get("Has Safety Valve")),
            "IncomingLines": [],
            "OutgoingLines": [],
            "ConnectedLines": [],
        })

    equipment_by_tag = {item["Tag"]: item for item in equipment_data}
    line_data = []
    unresolved_references = []

    for _, row in line_raw.iterrows():
        line_no = clean_value(row.get("배관 번호"))
        if line_no is None:
            continue

        from_tag = clean_value(row.get("From 장비"))
        to_tag = clean_value(row.get("To 장비"))

        design_pressure_kpag = to_float_or_none(row.get("설계압력 [kPag]"))
        design_pressure_barg = None if design_pressure_kpag is None else design_pressure_kpag / 100.0

        operating_temp_k = to_float_or_none(row.get("운전온도 [K]"))
        operating_temp_deg_c = None if operating_temp_k is None else operating_temp_k - 273.15

        line_item = {
            "LineNo": str(line_no).strip(),
            "FromTag": None if from_tag is None else str(from_tag).strip(),
            "ToTag": None if to_tag is None else str(to_tag).strip(),
            "DesignPressure": design_pressure_barg,
            "DesignPressureOriginal": design_pressure_kpag,
            "DesignPressureOriginalUnit": "kPag",
            "OperatingTemperature": operating_temp_deg_c,
            "OperatingTemperatureOriginal": operating_temp_k,
            "OperatingTemperatureOriginalUnit": "K",
            "Insulated": yes_no_to_bool(row.get("보온 여부")),
        }

        line_data.append(line_item)

        for col, tag_value, direction in [
            ("FromTag", line_item["FromTag"], "OutgoingLines"),
            ("ToTag", line_item["ToTag"], "IncomingLines"),
        ]:
            if tag_value in equipment_by_tag:
                equipment_by_tag[tag_value][direction].append(line_item["LineNo"])
                equipment_by_tag[tag_value]["ConnectedLines"].append(line_item["LineNo"])
            else:
                unresolved_references.append({
                    "LineNo": line_item["LineNo"],
                    "Column": col,
                    "MissingTag": tag_value,
                    "Reason": f"Line_List의 {col} 장비가 Equipment_List에 없음",
                })

    return equipment_data, line_data, unresolved_references

## 2. 과제3 md → 자연어 문장 추출

In [3]:
def extract_task3_sentences_from_md(md_path: Path) -> List[str]:
    text = md_path.read_text(encoding="utf-8")

    if "## 변환할 문장" in text:
        text = text.split("## 변환할 문장", 1)[1]
    if "## 여기서" in text:
        text = text.split("## 여기서", 1)[0]

    numbered_items = []
    current_num = None
    current_text = []

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith(">") or line.startswith("#") or line.startswith("```"):
            continue

        match = re.match(r"^(\d{1,2})\.\s+(.*)", line)

        if match:
            if current_num is not None:
                numbered_items.append((current_num, " ".join(current_text).strip()))
            current_num = int(match.group(1))
            current_text = [match.group(2).strip()]
        else:
            if current_num is not None and 1 <= current_num <= 12:
                if "일부 문장은" not in line:
                    current_text.append(line)

    if current_num is not None:
        numbered_items.append((current_num, " ".join(current_text).strip()))

    return [text.strip() for num, text in numbered_items if 1 <= num <= 12][:12]

## 3. 과제3 Validator

In [4]:
Expr = Union[list, tuple, str, int, float]

ALLOWED_FUNCS = {"allOf", "withSafetyValve", "prop", "isAtLeast", "isAtMost", "isEqual", "exists",
                 "matchesPattern", "times", "plus", "and", "or", "not", "forEvery"}
CLASS_NAMES = {"Equipment", "Vessel", "Tank", "Pump", "HeatExchanger", "Compressor"}
PROPERTY_NAMES = {"DesignPressure", "OperatingPressure", "DesignTemperature", "OperatingTemperature", "Tag", "Material"}
REGEX_NAMES = {"TAG_PATTERN"}


@dataclass
class ValidationResult:
    ok: bool
    errors: List[str]


def is_literal(x: Any) -> bool:
    return isinstance(x, (int, float, str)) and x != "x"


def as_tuple(expr: Any) -> Any:
    if isinstance(expr, list):
        return tuple(as_tuple(x) for x in expr)
    return expr


def func_name(expr: Any) -> str:
    expr = as_tuple(expr)
    if isinstance(expr, tuple) and len(expr) > 0:
        return expr[0]
    return ""


def validate_expr(expr: Expr, expected: str, path: str = "root") -> List[str]:
    expr = as_tuple(expr)
    errors = []

    if expected == "value":
        if is_literal(expr):
            return []
        if not isinstance(expr, tuple):
            return [f"{path}: value가 필요하지만 {expr!r} 입니다."]

        f = func_name(expr)

        if f == "prop":
            if len(expr) != 3:
                return [f"{path}: prop는 인자 2개가 필요합니다. 현재 {len(expr)-1}개"]
            _, target, prop_name = expr
            if target != "x":
                errors.append(f"{path}: prop의 첫 번째 인자는 반복 변수 x여야 합니다.")
            if prop_name not in PROPERTY_NAMES:
                errors.append(f"{path}: 허용되지 않은 속성명 {prop_name!r}")
            return errors

        if f in {"times", "plus"}:
            if len(expr) != 3:
                return [f"{path}: {f}는 인자 2개가 필요합니다. 현재 {len(expr)-1}개"]
            _, a, k = expr
            errors += validate_expr(a, "value", path + f".{f}[0]")
            if not isinstance(k, (int, float)):
                errors.append(f"{path}: {f}의 두 번째 인자는 숫자 리터럴이어야 합니다.")
            return errors

        return [f"{path}: value 위치에 허용되지 않은 함수 {f!r}가 사용되었습니다."]

    if expected == "selector":
        if not isinstance(expr, tuple):
            return [f"{path}: selector는 함수 호출이어야 합니다."]
        f = func_name(expr)
        if f not in {"allOf", "withSafetyValve"}:
            return [f"{path}: selector 위치에는 allOf/withSafetyValve만 올 수 있습니다. 현재 {f!r}"]
        if len(expr) != 2:
            return [f"{path}: {f}는 className 인자 1개가 필요합니다. 현재 {len(expr)-1}개"]
        if expr[1] not in CLASS_NAMES:
            errors.append(f"{path}: 허용되지 않은 className {expr[1]!r}")
        return errors

    if expected == "condition":
        if not isinstance(expr, tuple):
            return [f"{path}: condition은 함수 호출이어야 합니다."]
        f = func_name(expr)
        if f not in ALLOWED_FUNCS:
            return [f"{path}: 명세에 없는 함수 {f!r}가 사용되었습니다."]

        if f in {"isAtLeast", "isAtMost", "isEqual"}:
            if len(expr) != 3:
                return [f"{path}: {f}는 인자 2개가 필요합니다. 현재 {len(expr)-1}개"]
            errors += validate_expr(expr[1], "value", path + f".{f}[0]")
            errors += validate_expr(expr[2], "value", path + f".{f}[1]")
            return errors

        if f == "exists":
            if len(expr) != 2:
                return [f"{path}: exists는 인자 1개가 필요합니다. 현재 {len(expr)-1}개"]
            return validate_expr(expr[1], "value", path + ".exists[0]")

        if f == "matchesPattern":
            if len(expr) != 3:
                return [f"{path}: matchesPattern은 인자 2개가 필요합니다. 현재 {len(expr)-1}개"]
            errors += validate_expr(expr[1], "value", path + ".matchesPattern[0]")
            if expr[2] not in REGEX_NAMES:
                errors.append(f"{path}: 허용되지 않은 regexName {expr[2]!r}")
            return errors

        if f in {"and", "or"}:
            if len(expr) < 3:
                return [f"{path}: {f}는 최소 2개의 조건 인자가 필요합니다."]
            for i, child in enumerate(expr[1:]):
                errors += validate_expr(child, "condition", path + f".{f}[{i}]")
            return errors

        if f == "not":
            if len(expr) != 2:
                return [f"{path}: not은 인자 1개가 필요합니다. 현재 {len(expr)-1}개"]
            return validate_expr(expr[1], "condition", path + ".not[0]")

        return [f"{path}: condition 위치에 허용되지 않은 함수 {f!r}가 사용되었습니다."]

    return [f"{path}: 알 수 없는 expected type {expected!r}"]


def validate_rule(expr: Expr) -> ValidationResult:
    expr = as_tuple(expr)
    errors = []

    if not isinstance(expr, tuple) or len(expr) == 0:
        return ValidationResult(False, ["최상위 표현식은 forEvery(...) 함수 호출이어야 합니다."])

    if func_name(expr) != "forEvery":
        errors.append(f"최상위 함수는 forEvery여야 합니다. 현재 {func_name(expr)!r}")

    if func_name(expr) not in ALLOWED_FUNCS:
        errors.append(f"명세에 없는 최상위 함수 {func_name(expr)!r}가 사용되었습니다.")

    if len(expr) != 3:
        errors.append(f"forEvery는 target, condition 인자 2개가 필요합니다. 현재 {len(expr)-1}개")
        return ValidationResult(False, errors)

    errors += validate_expr(expr[1], "selector", "root.forEvery.target")
    errors += validate_expr(expr[2], "condition", "root.forEvery.condition")

    return ValidationResult(len(errors) == 0, errors)


def ast_to_string(expr: Expr) -> str:
    expr = as_tuple(expr)
    if isinstance(expr, tuple):
        return expr[0] + "(" + ", ".join(ast_to_string(x) for x in expr[1:]) + ")"
    if isinstance(expr, str):
        return f'"{expr}"' if expr != "x" else "x"
    return str(expr)

## 4. LLM AST 생성

In [5]:
def make_llm_prompt(sentence: str, feedback: str = "") -> str:
    feedback_block = ""
    if feedback:
        feedback_block = f"""
이전 출력은 validator를 통과하지 못했습니다.

실패 사유:
{feedback}

위 실패 사유를 반드시 반영해서 다시 생성하세요.
특히 실패 사유에 나온 함수명은 다시 사용하지 마세요.
"""

    return f"""
너는 산업 규칙 자연어 문장을 안전한 AST로 변환하는 변환기입니다.

반드시 아래 함수만 사용하세요.

허용 함수:
allOf, withSafetyValve, prop, isAtLeast, isAtMost, isEqual,
exists, matchesPattern, times, plus, and, or, not, forEvery

허용 className:
Equipment, Vessel, Tank, Pump, HeatExchanger, Compressor

허용 propertyName:
DesignPressure, OperatingPressure, DesignTemperature, OperatingTemperature, Tag, Material

허용 regexName:
TAG_PATTERN

중요 규칙:
1. 출력은 Python list 형태의 AST만 반환하세요.
2. 설명 문장, markdown 코드블록, 주석을 출력하지 마세요.
3. 명세에 없는 함수는 절대 만들지 마세요.
4. 최상위는 반드시 ["forEvery", target, condition] 형태여야 합니다.
5. "정의되어 있다", "정의되어 있어야 한다"는 반드시 ["exists", ["prop", "x", 속성명]] 으로 표현하세요.
6. "A이면 B"는 반드시 ["or", ["not", A], B]로 표현하세요.
7. x는 반드시 문자열 "x"로 표현하세요.
8. 숫자는 문자열이 아니라 숫자 리터럴로 출력하세요. 예: 1.1, 1.2, 30, 200, 240
9. 출력 전체는 반드시 하나의 Python list여야 합니다.

자연어 표현별 변환 예시:
- "모든 펌프" → ["allOf", "Pump"]
- "모든 용기" → ["allOf", "Vessel"]
- "안전밸브가 있는 모든 장비" → ["withSafetyValve", "Equipment"]

속성 표현별 변환 예시:
- "설계압력" → ["prop", "x", "DesignPressure"]
- "운전압력" → ["prop", "x", "OperatingPressure"]
- "설계온도" → ["prop", "x", "DesignTemperature"]

비교 표현별 변환 예시:
- "A는 B 이상이어야 한다" → ["isAtLeast", A, B]
- "A는 B 이하이어야 한다" → ["isAtMost", A, B]
- "A가 B와 같아야 한다" → ["isEqual", A, B]
- "A가 정의되어 있어야 한다" → ["exists", A]
- "A는 패턴과 일치해야 한다" → ["matchesPattern", A, "TAG_PATTERN"]

산술 표현별 변환 예시:
- "운전압력의 1.1배" → ["times", ["prop", "x", "OperatingPressure"], 1.1]
- "운전압력의 1.2배" → ["times", ["prop", "x", "OperatingPressure"], 1.2]
- "운전온도보다 30 이상 높다" → ["plus", ["prop", "x", "OperatingTemperature"], 30]

특수 규칙:
- "A 또는 B 중 하나"는 아래처럼 표현하세요.
  ["or",
    ["isEqual", ["prop", "x", "Material"], "A"],
    ["isEqual", ["prop", "x", "Material"], "B"]
  ]

- "운전온도가 A 이상인 모든 용기는 설계온도가 B 이상이어야 한다"는 조건문입니다.
  반드시 아래 구조처럼 A이면 B를 or(not(A), B)로 표현하세요.
  ["forEvery",
    ["allOf", "Vessel"],
    ["or",
      ["not", ["isAtLeast", ["prop", "x", "OperatingTemperature"], A]],
      ["isAtLeast", ["prop", "x", "DesignTemperature"], B]
    ]
  ]

정답 형식 예시:
["forEvery", ["withSafetyValve", "Equipment"], ["exists", ["prop", "x", "DesignPressure"]]]

{feedback_block}

변환할 자연어 문장:
{sentence}
""".strip()


def extract_json_array(text: str) -> List[Any]:
    text = text.strip()
    text = text.replace("```python", "").replace("```json", "").replace("```", "").strip()
    start = text.find("[")
    end = text.rfind("]")

    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"LLM 응답에서 AST list를 찾지 못했습니다: {text}")

    candidate = text[start:end + 1]

    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        import ast
        return ast.literal_eval(candidate)


def call_real_llm(sentence: str, feedback: str = "") -> List[Any]:
    try:
        from openai import OpenAI
    except ImportError as exc:
        raise ImportError("openai 패키지가 필요합니다. pip install openai 를 실행하세요.") from exc

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    prompt = make_llm_prompt(sentence, feedback)

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
    )

    return extract_json_array(response.output_text)


def deterministic_ast_for_mock(sentence: str) -> List[Any]:
    s = sentence.strip()

    if "모든 용기" in s and "1.1배" in s and "설계압력" in s:
        return ["forEvery", ["allOf", "Vessel"], ["isAtLeast", ["prop", "x", "DesignPressure"], ["times", ["prop", "x", "OperatingPressure"], 1.1]]]
    if "안전밸브가 있는 모든 장비" in s and "정의" in s and "1.2배" not in s:
        return ["forEvery", ["withSafetyValve", "Equipment"], ["exists", ["prop", "x", "DesignPressure"]]]
    if "모든 열교환기" in s and "설계온도" in s and "30" in s:
        return ["forEvery", ["allOf", "HeatExchanger"], ["isAtLeast", ["prop", "x", "DesignTemperature"], ["plus", ["prop", "x", "OperatingTemperature"], 30]]]
    if "모든 장비 태그" in s:
        return ["forEvery", ["allOf", "Equipment"], ["matchesPattern", ["prop", "x", "Tag"], "TAG_PATTERN"]]
    if "모든 펌프" in s and "재질" in s and "설계압력" in s and "정의" in s:
        return ["forEvery", ["allOf", "Pump"], ["and", ["exists", ["prop", "x", "Material"]], ["exists", ["prop", "x", "DesignPressure"]]]]
    if "어떤 탱크도" in s and "2 barg" in s:
        return ["forEvery", ["allOf", "Tank"], ["isAtMost", ["prop", "x", "OperatingPressure"], 2]]
    if "운전온도가 200 이상" in s and "설계온도가 240 이상" in s:
        return ["forEvery", ["allOf", "Vessel"], ["or", ["not", ["isAtLeast", ["prop", "x", "OperatingTemperature"], 200]], ["isAtLeast", ["prop", "x", "DesignTemperature"], 240]]]
    if "안전밸브가 있는 모든 장비" in s and "1.2배" in s:
        return ["forEvery", ["withSafetyValve", "Equipment"], ["isAtLeast", ["prop", "x", "DesignPressure"], ["times", ["prop", "x", "OperatingPressure"], 1.2]]]
    if "모든 압축기" in s and "1.1배" in s and "30" in s:
        return ["forEvery", ["allOf", "Compressor"], ["and", ["isAtLeast", ["prop", "x", "DesignPressure"], ["times", ["prop", "x", "OperatingPressure"], 1.1]], ["isAtLeast", ["prop", "x", "DesignTemperature"], ["plus", ["prop", "x", "OperatingTemperature"], 30]]]]
    if "모든 열교환기의 재질" in s and "SS304" in s and "SS316" in s:
        return ["forEvery", ["allOf", "HeatExchanger"], ["or", ["isEqual", ["prop", "x", "Material"], "SS304"], ["isEqual", ["prop", "x", "Material"], "SS316"]]]
    if "어떤 장비도" in s and "설계온도가 운전온도보다 낮아서는 안" in s:
        return ["forEvery", ["allOf", "Equipment"], ["isAtLeast", ["prop", "x", "DesignTemperature"], ["prop", "x", "OperatingTemperature"]]]
    if "모든 펌프 태그" in s and "태그 패턴" in s:
        return ["forEvery", ["allOf", "Pump"], ["and", ["matchesPattern", ["prop", "x", "Tag"], "TAG_PATTERN"], ["exists", ["prop", "x", "DesignPressure"]]]]

    raise ValueError(f"mock LLM이 처리하지 못한 문장입니다: {sentence}")


def call_mock_llm(sentence: str, feedback: str = "", attempt: int = 1) -> List[Any]:
    if attempt == 1:
        if "정의" in sentence:
            return ["forEvery", ["allOf", "Pump"], ["defined", ["prop", "x", "DesignPressure"]]]
        if "1.1배" in sentence:
            return ["forEvery", ["allOf", "Vessel"], ["greaterThan", ["prop", "x", "DesignPressure"], ["times", ["prop", "x", "OperatingPressure"], 1.1]]]
        if "태그 패턴" in sentence:
            return ["forEvery", ["allOf", "Equipment"], ["matches", ["prop", "x", "Tag"], "TAG_PATTERN"]]

    return deterministic_ast_for_mock(sentence)


def call_llm(sentence: str, feedback: str = "", attempt: int = 1) -> List[Any]:
    if USE_REAL_LLM:
        return call_real_llm(sentence, feedback)

    return call_mock_llm(sentence, feedback, attempt)


def llm_generate_rule_with_gate_loop(sentence: str, rule_id: str, max_retry: int = MAX_RETRY) -> Dict[str, Any]:
    feedback = ""
    attempts_log = []
    last_ast = None
    last_errors = []

    for attempt in range(1, max_retry + 1):
        ast_expr = call_llm(sentence, feedback, attempt)
        last_ast = ast_expr
        result = validate_rule(ast_expr)

        attempts_log.append({
            "attempt": attempt,
            "ast": ast_expr,
            "ast_text": ast_to_string(ast_expr),
            "ok": result.ok,
            "errors": result.errors,
        })

        if result.ok:
            return {
                "rule_id": rule_id,
                "source_sentence": sentence,
                "ast": ast_expr,
                "status": "pass",
                "attempts": attempt,
                "generation_log": attempts_log,
            }

        last_errors = result.errors
        feedback = "\n".join(result.errors)

    return {
        "rule_id": rule_id,
        "source_sentence": sentence,
        "ast": last_ast,
        "status": "fail",
        "attempts": max_retry,
        "errors": last_errors,
        "generation_log": attempts_log,
    }

## 5. AST 실행기

In [6]:
def list_to_tuple(obj: Any) -> Any:
    if isinstance(obj, list):
        return tuple(list_to_tuple(x) for x in obj)
    return obj


def select_targets(selector: Expr, data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    selector = list_to_tuple(selector)
    func = selector[0]

    if func == "allOf":
        class_name = selector[1]
        if class_name == "Equipment":
            return data
        return [row for row in data if row.get("ClassName") == class_name]

    if func == "withSafetyValve":
        class_name = selector[1]
        rows = [row for row in data if row.get("HasSafetyValve") is True]
        if class_name == "Equipment":
            return rows
        return [row for row in rows if row.get("ClassName") == class_name]

    raise ValueError(f"지원하지 않는 selector 함수: {func}")


def eval_value(expr: Expr, x: Dict[str, Any]) -> Any:
    expr = list_to_tuple(expr)

    if isinstance(expr, (int, float, str)) and expr != "x":
        return expr

    func = expr[0]

    if func == "prop":
        _, target, property_name = expr
        if target != "x":
            raise ValueError("prop의 첫 번째 인자는 x여야 합니다.")
        return x.get(property_name)

    if func == "times":
        _, a, k = expr
        value = eval_value(a, x)
        return None if value is None else value * k

    if func == "plus":
        _, a, k = expr
        value = eval_value(a, x)
        return None if value is None else value + k

    raise ValueError(f"지원하지 않는 value 함수: {func}")


def matches_tag_pattern(tag: Any) -> bool:
    if tag is None:
        return False
    return re.match(r"^[A-Za-z]+-\d{3}[A-Za-z]?$", str(tag)) is not None


def eval_condition(expr: Expr, x: Dict[str, Any]) -> bool:
    expr = list_to_tuple(expr)
    func = expr[0]

    if func == "exists":
        value = eval_value(expr[1], x)
        return value is not None and value != ""

    if func == "isAtLeast":
        left = eval_value(expr[1], x)
        right = eval_value(expr[2], x)
        if left is None or right is None:
            return False
        return left >= right

    if func == "isAtMost":
        left = eval_value(expr[1], x)
        right = eval_value(expr[2], x)
        if left is None or right is None:
            return False
        return left <= right

    if func == "isEqual":
        return eval_value(expr[1], x) == eval_value(expr[2], x)

    if func == "matchesPattern":
        value = eval_value(expr[1], x)
        regex_name = expr[2]
        if regex_name == "TAG_PATTERN":
            return matches_tag_pattern(value)
        raise ValueError(f"지원하지 않는 regexName: {regex_name}")

    if func == "and":
        return all(eval_condition(child, x) for child in expr[1:])

    if func == "or":
        return any(eval_condition(child, x) for child in expr[1:])

    if func == "not":
        return not eval_condition(expr[1], x)

    raise ValueError(f"지원하지 않는 condition 함수: {func}")


def evaluate_rule(rule: Dict[str, Any], equipment_data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    ast_expr = list_to_tuple(rule["ast"])

    if ast_expr[0] != "forEvery":
        raise ValueError("최상위 함수는 forEvery여야 합니다.")

    _, selector, condition = ast_expr
    targets = select_targets(selector, equipment_data)
    violations = []

    for row in targets:
        if not eval_condition(condition, row):
            violations.append({
                "rule_id": rule["rule_id"],
                "source_sentence": rule["source_sentence"],
                "tag": row.get("Tag"),
                "class_name": row.get("ClassName"),
                "reason": "조건을 만족하지 않음",
            })

    return violations


def build_violation_report(equipment_data: List[Dict[str, Any]], rules: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    all_violations = []
    for rule in rules:
        all_violations.extend(evaluate_rule(rule, equipment_data))
    return all_violations

## 6. 저장

In [7]:
def save_json(path: Path, data: Any) -> None:
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def flatten_generation_log(rules: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    rows = []
    for rule in rules:
        for item in rule.get("generation_log", []):
            rows.append({
                "rule_id": rule["rule_id"],
                "source_sentence": rule["source_sentence"],
                "attempt": item["attempt"],
                "ok": item["ok"],
                "errors": " | ".join(item["errors"]),
                "ast": item["ast_text"],
            })
    return rows


def save_outputs(equipment_data, line_data, unresolved_references, rules, valid_rules, validation_rows, violation_report):
    OUT_DIR.mkdir(exist_ok=True)

    save_json(OUT_DIR / "equipment_data.json", equipment_data)
    save_json(OUT_DIR / "line_data.json", line_data)
    save_json(OUT_DIR / "llm_rules.json", rules)
    save_json(OUT_DIR / "valid_rules.json", valid_rules)

    pd.DataFrame(equipment_data).to_csv(OUT_DIR / "equipment_data.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(line_data).to_csv(OUT_DIR / "line_data.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(unresolved_references).to_csv(OUT_DIR / "unresolved_line_references.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(flatten_generation_log(rules)).to_csv(OUT_DIR / "llm_generation_log.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(validation_rows).to_csv(OUT_DIR / "rule_validation_report.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(violation_report).to_csv(OUT_DIR / "violation_report.csv", index=False, encoding="utf-8-sig")

## 7. main

In [8]:
def main() -> None:
    print("=== LLM 과제2+과제3 통합 파이프라인 시작 ===")
    print("EXCEL_PATH:", EXCEL_PATH)
    print("MD_PATH:", MD_PATH)
    print("USE_REAL_LLM:", USE_REAL_LLM)

    equipment_data, line_data, unresolved_references = build_equipment_data(EXCEL_PATH)
    sentences = extract_task3_sentences_from_md(MD_PATH)

    print("\n데이터 생성")
    print("- equipment_data:", len(equipment_data))
    print("- line_data:", len(line_data))
    print("- unresolved_references:", len(unresolved_references))
    print("- sentences:", len(sentences))

    rules = []

    for i, sentence in enumerate(sentences, start=1):
        rule_id = f"LLM_T3_{i:02d}"
        rule = llm_generate_rule_with_gate_loop(sentence, rule_id)
        rules.append(rule)
        print(f"{rule_id}: {rule['status']} / attempts={rule['attempts']}")

    validation_rows = []
    valid_rules = []

    for rule in rules:
        result = validate_rule(rule["ast"])
        validation_rows.append({
            "rule_id": rule["rule_id"],
            "status": rule["status"],
            "ok": result.ok,
            "errors": " | ".join(result.errors),
            "attempts": rule.get("attempts"),
            "ast": ast_to_string(rule["ast"]),
        })

        if result.ok:
            valid_rules.append(rule)

    violation_report = build_violation_report(equipment_data, valid_rules)

    print("\n검증 결과")
    print("- 전체 rules:", len(rules))
    print("- valid_rules:", len(valid_rules))
    print("- violation_report:", len(violation_report))

    save_outputs(equipment_data, line_data, unresolved_references, rules, valid_rules, validation_rows, violation_report)

    print("\n저장 완료:", OUT_DIR.resolve())
    for p in OUT_DIR.iterdir():
        print("-", p.name)

In [9]:
main()

=== LLM 과제2+과제3 통합 파이프라인 시작 ===
EXCEL_PATH: consistency_check.xlsx
MD_PATH: task3_nl_to_modules.md
USE_REAL_LLM: True

데이터 생성
- equipment_data: 10
- line_data: 8
- unresolved_references: 1
- sentences: 12
LLM_T3_01: pass / attempts=1
LLM_T3_02: pass / attempts=1
LLM_T3_03: pass / attempts=1
LLM_T3_04: pass / attempts=1
LLM_T3_05: pass / attempts=1
LLM_T3_06: pass / attempts=1
LLM_T3_07: pass / attempts=1
LLM_T3_08: pass / attempts=1
LLM_T3_09: pass / attempts=1
LLM_T3_10: pass / attempts=1
LLM_T3_11: pass / attempts=1
LLM_T3_12: pass / attempts=1

검증 결과
- 전체 rules: 12
- valid_rules: 12
- violation_report: 6

저장 완료: C:\Users\Administrator\Desktop\technical_review\llm_pipeline_output
- equipment_data.csv
- equipment_data.json
- line_data.csv
- line_data.json
- llm_generation_log.csv
- llm_rules.json
- rule_validation_report.csv
- unresolved_line_references.csv
- valid_rules.json
- violation_report.csv
